# Task 1 — Exploratory Data Analysis on Retail Sales Data

**Internship Project | Data Analytics**

### Objective
Perform Exploratory Data Analysis (EDA) on retail sales data to identify sales patterns, customer behaviour, product performance and actionable business insights.

### Tech Stack
Python, pandas, matplotlib, seaborn, Jupyter Notebook

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

: 

## 2. Load Dataset

The uploaded retail sales CSV contains transaction-level information such as date, customer, gender, age, product category, quantity, unit price and total transaction amount.

In [ ]:
# Update the path if the CSV is moved to another folder
file_path = "vicky.int retail_sales_dataset.csv"
df = pd.read_csv(file_path)

df.head()

## 3. Initial Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumn data types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

### Observation
The dataset contains **1,000 transactions and 9 columns**. All columns are complete with **no missing values**. The dataset includes both numerical and categorical variables, making it suitable for sales, customer and product-level EDA.

## 4. Data Preparation

The `Date` column is converted to datetime format. Additional fields are created for monthly, quarterly and age-group analysis.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)

df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 35, 45, 55, 200],
    labels=["18–25", "26–35", "36–45", "46–55", "56+"],
    include_lowest=True
)

df.head()

## 5. Descriptive Statistics

The following table calculates the **mean, median, mode and standard deviation** for every numerical variable.

In [ ]:
numerical_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]

statistics = pd.DataFrame({
    "Mean": df[numerical_cols].mean(),
    "Median": df[numerical_cols].median(),
    "Mode": [df[col].mode().iloc[0] for col in numerical_cols],
    "Standard Deviation": df[numerical_cols].std()
})

statistics.round(2)

### Observation
The descriptive statistics show the central tendency and variability of customer age, purchase quantity, unit price and transaction value. Comparing mean and median helps identify whether a variable is influenced by unusually high or low observations.

## 6. Monthly Sales Trend

In [ ]:
monthly_sales = df.groupby("Month")["Total Amount"].sum()

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Observation
Monthly sales fluctuate across the year, showing that customer demand is not uniform. The highest-sales month can be used by management for inventory and staffing planning, while weaker months can be targeted with promotional campaigns.

## 7. Quarterly Sales Trend

In [ ]:
quarterly_sales = df.groupby("Quarter")["Total Amount"].sum()

plt.figure(figsize=(10, 5))
plt.plot(quarterly_sales.index, quarterly_sales.values, marker="o")
plt.title("Quarterly Sales Trend")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

### Observation
Quarterly aggregation smooths out individual monthly fluctuations and makes broader sales patterns easier to compare. The strongest quarter represents the period where the business generated the greatest revenue.

## 8. Customer Demographics — Age Groups

In [ ]:
age_distribution = df["Age Group"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
age_distribution.plot(kind="bar")
plt.title("Customer Transactions by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Observation
The age-group distribution identifies which customer segments contribute most transaction activity. The largest segment can be prioritized for targeted marketing, while smaller segments may represent opportunities for customer acquisition.

## 9. Customer Demographics — Gender Breakdown

In [ ]:
gender_counts = df["Gender"].value_counts()

plt.figure(figsize=(7, 5))
gender_counts.plot(kind="bar")
plt.title("Gender Distribution of Customers")
plt.xlabel("Gender")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Observation
The gender breakdown shows how transactions are distributed across customer genders. This can support more focused customer segmentation, provided marketing decisions are based on broader behavioural evidence rather than gender alone.

## 10. Product Analysis — Top 10 Best-Selling Products

This dataset contains product **categories rather than individual product names**, so the top-10 analysis is performed at the product-category level using total quantity sold. This is an important data limitation to document rather than invent product names.

In [ ]:
top_categories_quantity = (
    df.groupby("Product Category")["Quantity"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(9, 5))
top_categories_quantity.plot(kind="bar")
plt.title("Best-Selling Product Categories by Quantity")
plt.xlabel("Product Category")
plt.ylabel("Units Sold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Observation
The category-level ranking identifies which product categories have the highest unit demand. These categories should receive close attention in inventory planning because shortages in high-volume categories can directly affect sales.

## 11. Revenue by Product Category

In [ ]:
category_revenue = (
    df.groupby("Product Category")["Total Amount"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
category_revenue.plot(kind="bar")
plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Observation
Revenue contribution differs across product categories. Management should prioritize high-revenue categories while also checking their margins, customer retention and inventory requirements before increasing promotional spending.

## 12. Correlation Heatmap

In [ ]:
numerical_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

### Observation
The heatmap shows the strength and direction of linear relationships between numerical variables. In particular, transaction amount is expected to be closely related to quantity and price because total amount is constructed from purchase quantity and unit price. Correlation indicates association, not causation.

## 13. Additional Visualization — Age Group vs Average Transaction Value

This visualization looks beyond transaction counts. It checks whether different age groups have different **average spending per transaction**, which can reveal a less obvious customer-value pattern.

In [ ]:
age_group_spend = (
    df.groupby("Age Group", observed=False)["Total Amount"]
      .mean()
      .sort_index()
)

plt.figure(figsize=(9, 5))
age_group_spend.plot(kind="bar")
plt.title("Average Transaction Value by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Transaction Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Observation
Comparing average transaction value with transaction volume helps distinguish between **high-frequency customers** and **higher-value customers**. A segment with fewer transactions but a higher average transaction value may deserve premium offers or retention strategies.

# 14. Key Business Insights

Based on the analysis, the following areas should be highlighted:

1. **Sales seasonality:** Monthly and quarterly trends identify stronger and weaker periods.
2. **Customer segmentation:** Age-group and gender distributions show the composition of the customer base.
3. **Category performance:** Quantity and revenue rankings reveal which product categories drive demand and revenue.
4. **Customer value:** Average transaction value by age group provides an additional view beyond transaction counts.
5. **Variable relationships:** The correlation matrix helps identify numerical relationships that may be useful for further analysis.

# 15. Actionable Business Recommendations

### Recommendation 1 — Improve inventory planning
Use the monthly and quarterly sales trends to increase stock availability before high-demand periods and avoid unnecessary inventory during weaker periods.

### Recommendation 2 — Prioritize high-performing categories
Allocate more merchandising and promotional attention to categories with strong revenue and quantity performance, while monitoring profitability before scaling discounts.

### Recommendation 3 — Use customer-value segmentation
Combine age-group transaction volume with average transaction value. Design different campaigns for high-frequency customers and high-value customers rather than treating every segment equally.

### Recommendation 4 — Review pricing and promotion strategy
Use the relationship between quantity, price and transaction value as a starting point for deeper pricing analysis. Future analysis should add discount and profit data if available.

### Recommendation 5 — Build a more detailed product dataset
The current file contains product categories but not individual product names. Adding SKU/product-level information would allow true **Top 10 Product** analysis and more precise inventory decisions.

# 16. Conclusion

The EDA provides a structured view of retail sales performance across time, customers and product categories. The analysis identifies sales fluctuations, customer demographic patterns and category-level revenue and demand differences.

The most useful next step would be to extend the analysis with **profit, discount, product/SKU and customer repeat-purchase data**. This would allow the business to move from descriptive analysis toward profitability analysis, customer lifetime value and predictive decision-making.